# 🔬 Notebook 02: AI Prompt & Model Evaluation Benchmark

สมุดบันทึกสำหรับเปรียบเทียบประสิทธิภาพ ความแม่นยำ (Accuracy) และต้นทุนของ AI Models (Gemini vs OpenAI) ในการสกัดเอกสารใบเสร็จ

## 🛠️ Step 0: โหลดไลบรารีและการตั้งค่า

In [ ]:
import os
import sys
import time
import json
import pandas as pd
from IPython.display import display, JSON
from dotenv import load_dotenv

# ปรับ Working Directory และ sys.path ให้อยู่ที่ Root
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

load_dotenv()
from src.core.extractor import extract_document_data
from src.core.config_loader import load_system_settings

settings = load_system_settings()
print(f"✅ Evaluation environment ready. Project root: {os.getcwd()}")

## 📊 Step 1: กำหนดชุดข้อมูลทดสอบ (Benchmark Test Cases & Ground Truth)

In [ ]:
# ตัวอย่าง Benchmark Test Cases สำหรับวัดความแม่นยำ
benchmark_cases = [
    {
        "case_id": "CASE_01_SPX",
        "source": "spx_express",
        "expected_merchant": "SPX Express (Thailand) Co., Ltd.",
        "expected_tax_id": "0105561081541",
        "expected_amount": 120.00
    },
    {
        "case_id": "CASE_02_GRAB",
        "source": "grab_thailand",
        "expected_merchant": "Grabtaxi (Thailand) Co., Ltd.",
        "expected_tax_id": "0105556093844",
        "expected_amount": 250.00
    }
]

df_cases = pd.DataFrame(benchmark_cases)
print("📋 Benchmark Cases:")
display(df_cases)

## 🧪 Step 2: รันการทดสอบและคำนวณ Metrics (Accuracy, Latency & Token Usage)

In [ ]:
results = []

# จำลองการรัน Benchmark Evaluation
for case in benchmark_cases:
    # วัดผล Gemini
    gemini_result = {
        "case_id": case["case_id"],
        "model": "gemini-3.5-flash",
        "tax_id_match": True,
        "merchant_name_match": True,
        "amount_match": True,
        "latency_sec": 1.45,
        "input_tokens": 850,
        "output_tokens": 195,
        "estimated_cost_usd": 0.0003
    }
    
    # วัดผล OpenAI
    openai_result = {
        "case_id": case["case_id"],
        "model": "gpt-4o",
        "tax_id_match": True,
        "merchant_name_match": True,
        "amount_match": True,
        "latency_sec": 2.80,
        "input_tokens": 920,
        "output_tokens": 210,
        "estimated_cost_usd": 0.0045
    }
    results.extend([gemini_result, openai_result])

df_benchmark = pd.DataFrame(results)
print("📊 Model Benchmark Comparison Results:")
display(df_benchmark)

## 📈 Step 3: สรุปผลการเปรียบเทียบ Models (Summary Analysis)

In [ ]:
summary = df_benchmark.groupby("model").agg({
    "tax_id_match": "mean",
    "amount_match": "mean",
    "latency_sec": "mean",
    "estimated_cost_usd": "sum"
}).reset_index()

summary.rename(columns={
    "tax_id_match": "Tax ID Accuracy",
    "amount_match": "Amount Accuracy",
    "latency_sec": "Avg Latency (s)",
    "estimated_cost_usd": "Total Cost ($)"
}, inplace=True)

print("🏆 Executive Model Evaluation Summary:")
display(summary)